# DA6401 Assignment 1 – W&B Demo Notebook

This notebook demonstrates:
1. **Data Exploration** – sample images logged to W&B
2. **Hyperparameter Sweep** – W&B sweep over key hyperparameters
3. **Gradient Analysis** – logging gradient norms per layer
4. **Dead Neuron Investigation** – activation distributions
5. **Confusion Matrix** – best model evaluation

> Make sure you run `pip install wandb keras scikit-learn matplotlib numpy` before executing.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import wandb

from ann.neural_network import NeuralNetwork
from ann.activations import softmax
from utils.data_loader import load_dataset, get_class_names

## 1. Data Exploration

In [ ]:
DATASET = 'fashion_mnist'
X_train, y_train, X_val, y_val, X_test, y_test = load_dataset(DATASET)
class_names = get_class_names(DATASET)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# Log 5 samples per class to W&B
run = wandb.init(project='da6401_assignment1', name='data_exploration')

columns = ['label', 'class_name', 'image']
table = wandb.Table(columns=columns)

for cls in range(10):
    idxs = np.where(y_train == cls)[0][:5]
    for idx in idxs:
        img = X_train[idx].reshape(28, 28)
        table.add_data(cls, class_names[cls], wandb.Image(img))

run.log({'sample_images': table})
run.finish()
print('Logged sample images to W&B.')

## 2. Hyperparameter Sweep

In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'optimizer':      {'values': ['sgd', 'momentum', 'nag', 'rmsprop']},
        'learning_rate':  {'values': [0.1, 0.01, 0.001, 0.0001]},
        'num_layers':     {'values': [2, 3, 4]},
        'hidden_size':    {'values': [64, 128]},
        'activation':     {'values': ['sigmoid', 'tanh', 'relu']},
        'weight_init':    {'values': ['random', 'xavier']},
        'weight_decay':   {'values': [0.0, 0.0001, 0.001]},
        'batch_size':     {'values': [32, 64, 128]},
    }
}

def sweep_train():
    run = wandb.init()
    cfg = wandb.config

    model = NeuralNetwork(dict(
        num_layers=cfg.num_layers,
        hidden_size=cfg.hidden_size,
        activation=cfg.activation,
        weight_init=cfg.weight_init,
        loss='cross_entropy',
        weight_decay=cfg.weight_decay,
    ))

    train_cfg = dict(
        epochs=10,
        batch_size=cfg.batch_size,
        optimizer=cfg.optimizer,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    model.fit(X_train, y_train, X_val, y_val, train_cfg, wandb_run=run)
    run.finish()

sweep_id = wandb.sweep(sweep_config, project='da6401_assignment1')
print(f'Sweep ID: {sweep_id}')
# Uncomment to run 100 agents:
# wandb.agent(sweep_id, function=sweep_train, count=100)

## 3. Optimizer Comparison (Q2.3)

In [ ]:
optimizers = ['sgd', 'momentum', 'nag', 'rmsprop']
results = {}

for opt in optimizers:
    run = wandb.init(project='da6401_assignment1', name=f'optimizer_{opt}',
                     tags=['optimizer_comparison'])

    model = NeuralNetwork(dict(
        num_layers=3, hidden_size=128, activation='relu',
        weight_init='xavier', loss='cross_entropy', weight_decay=0.0,
    ))
    train_cfg = dict(epochs=10, batch_size=32, optimizer=opt,
                     learning_rate=0.001, weight_decay=0.0)
    model.fit(X_train, y_train, X_val, y_val, train_cfg, wandb_run=run)
    run.finish()
    print(f'Done: {opt}')

## 4. Vanishing Gradient Analysis (Q2.4)

In [ ]:
from ann.objective_functions import get_loss
from ann.optimizers import get_optimizer

def train_log_gradients(activation, num_layers=4, epochs=10, run_name=None):
    """Train a model and log first-layer gradient norms to W&B."""
    run = wandb.init(project='da6401_assignment1',
                     name=run_name or f'grad_analysis_{activation}',
                     tags=['gradient_analysis'])

    model = NeuralNetwork(dict(
        num_layers=num_layers, hidden_size=128, activation=activation,
        weight_init='xavier', loss='cross_entropy', weight_decay=0.0,
    ))

    loss_fn, loss_grad = get_loss('cross_entropy')
    optimizer = get_optimizer('rmsprop', 0.001, 0.0)

    step = 0
    for epoch in range(epochs):
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), 32):
            batch_idx = idx[start:start+32]
            Xb, yb = X_train[batch_idx], y_train[batch_idx]
            logits = model.forward(Xb)
            model.backward(logits, yb)
            optimizer.update(model.layers)

            grad_norm = np.linalg.norm(model.layers[0].grad_W)
            run.log({'first_layer_grad_norm': grad_norm, 'step': step})
            step += 1

    run.finish()

train_log_gradients('sigmoid', run_name='sigmoid_grad_norm')
train_log_gradients('relu',    run_name='relu_grad_norm')

## 5. Weight Initialisation Symmetry (Q2.9)

In [ ]:
def train_log_neuron_gradients(weight_init, run_name, n_neurons=5, n_iters=50):
    """Log individual neuron gradients for the first hidden layer."""
    run = wandb.init(project='da6401_assignment1', name=run_name,
                     tags=['weight_init_symmetry'])

    model = NeuralNetwork(dict(
        num_layers=3, hidden_size=128, activation='relu',
        weight_init=weight_init, loss='cross_entropy', weight_decay=0.0,
    ))

    # Zero init override
    if weight_init == 'zeros':
        for layer in model.layers:
            layer.W = np.zeros_like(layer.W)
            layer.b = np.zeros_like(layer.b)

    loss_fn, loss_grad = get_loss('cross_entropy')
    optimizer = get_optimizer('sgd', 0.01, 0.0)

    for step in range(n_iters):
        idx = np.random.choice(len(X_train), 32, replace=False)
        Xb, yb = X_train[idx], y_train[idx]
        logits = model.forward(Xb)
        model.backward(logits, yb)
        optimizer.update(model.layers)

        log_dict = {'step': step}
        for n in range(n_neurons):
            log_dict[f'neuron_{n}_grad_norm'] = np.linalg.norm(
                model.layers[0].grad_W[:, n]
            )
        run.log(log_dict)

    run.finish()

train_log_neuron_gradients('zeros',  run_name='zeros_init_symmetry')
train_log_neuron_gradients('xavier', run_name='xavier_init_symmetry')

## 6. Confusion Matrix (Q2.8)

In [ ]:
import json
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load best model
weights = np.load('../src/best_model.npy', allow_pickle=True).item()
with open('../src/best_config.json') as f:
    cfg = json.load(f)

model = NeuralNetwork(cfg)
model.set_weights(weights)

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
plt.title('Confusion Matrix – Best Model')
plt.tight_layout()

run = wandb.init(project='da6401_assignment1', name='confusion_matrix')
run.log({'confusion_matrix': wandb.Image(fig)})
run.finish()
plt.show()